# MSC Figure 6 — Perez end-to-end reproducible pipeline

This Google Colab notebook reproduces the Perez et al. extension of Figure 6 from public data through final plotting tables.

## Locked analysis

- Source: Perez et al. 2021, GEO `GSE157204`, published soma–dendrite labels from Supplementary File 5.
- Annotation: Ensembl release 116, rat assembly GRCr8.
- Panel: 250 dendrite-enriched and 250 soma-enriched protein-coding genes, selected and matched before sequence composition is calculated.
- Matching: total normalized expression and log canonical 3′UTR length.
- Original sequence proxy: `primary_g_rg4 = z((z(G fraction) + z(canonical rG4 density))/2)`.
- Decomposition: `g_only`.
- Mapping control: shuffled transcript-to-coordinate mapping.
- Primary PSD calibration: `tau2_ms`, 100 matched seeds.
- Sensitivities: mobile fraction and elastic modulus, 30 matched seeds each.
- Outputs: full-precision CSV/CSV.GZ files for R, SHA-256 manifests, resumable checkpoints, and final audit.

The sequence coordinate remains a proxy rather than a direct measurement of RNP material state. All docking and productive-capture outputs are computational predictions.

In [ ]:
# 1. Runtime-local paths, model upload, dependencies and locked constants
# Drive is NOT required. Model and config are uploaded here, at the very start,
# so the run cannot fail later in section 10.
!pip -q install pyyaml statsmodels scipy pandas numpy

from pathlib import Path
import os, sys, re, gzip, json, yaml, shutil, hashlib, datetime, zipfile, importlib, warnings
from collections import defaultdict
import requests
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import milp, LinearConstraint, Bounds, linear_sum_assignment
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from google.colab import files

warnings.filterwarnings('ignore')

ROOT = Path('/content/msc_figure6_perez/output')

USE_DRIVE_MIRROR = False
DRIVE_MIRROR = Path('/content/drive/MyDrive/MSC_Results_0726/figure6_perez_harmonised')
if USE_DRIVE_MIRROR:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MIRROR.mkdir(parents=True, exist_ok=True)

DIRS = {
    'raw': ROOT / 'raw',
    'annotation': ROOT / 'annotation',
    'manifests': ROOT / 'manifests',
    'panels': ROOT / 'panels',
    'scores': ROOT / 'scores',
    'model': ROOT / 'model',
    'checkpoints': ROOT / 'checkpoints',
    'data': ROOT / 'data',
    'statistics': ROOT / 'statistics',
    'plotting': ROOT / 'plotting',
    'logs': ROOT / 'logs',
}
for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

R_INPUT = ROOT / '10_r_input'
R_INPUT.mkdir(parents=True, exist_ok=True)

# --- Locked constants. Names and values are identical to the manuscript run. ---
FLOAT_FORMAT = '%.17g'
MIN_UTR_LENGTH = 100
N_MATCHED_PER_GROUP = 250
PANEL_SEED = 20260713
PRIMARY_N_SEEDS = 100
SENSITIVITY_N_SEEDS = 30
CHUNK_SIZE = 10
N_BOOT = 20000
N_PERMUTATIONS = 20000
PARAMETER_SEED_BASE = 20000
SIMULATION_SEED_BASE = 20013

# --- Harmonisation additions ---
DATASET_ID = 'Perez_2021_GSE157204'
SPECIES = 'Rattus_norvegicus'
SELECTION_SOURCE = 'published_supplementary_file_5'
ANALYSIS_UNIT = 'gene_canonical_3pUTR'
MODEL_VARIANT = 'full_msc_feedback'
TRIM_QUANTILE = 0.95


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()


# --- Upload the MSC model and config now ---
MODEL_PATH = Path('/content/msc_model_feedback.py')
CFG_PATH = Path('/content/config_msc_feedback.yaml')

if not MODEL_PATH.exists() or not CFG_PATH.exists():
    print('Upload msc_model_feedback.py and config_msc_feedback.yaml now.')
    uploaded = files.upload()
    for name in uploaded:
        source = Path('/content') / name
        if name.endswith('.py'):
            shutil.copy2(source, MODEL_PATH)
        elif name.endswith(('.yaml', '.yml')):
            shutil.copy2(source, CFG_PATH)

if not MODEL_PATH.exists() or not CFG_PATH.exists():
    raise FileNotFoundError(
        'msc_model_feedback.py and config_msc_feedback.yaml are required. '
        'Re-run this cell and upload both files.'
    )

print('Dataset    :', DATASET_ID)
print('Output root:', ROOT)
print('Model      :', sha256_file(MODEL_PATH)[:16])
print('Config     :', sha256_file(CFG_PATH)[:16])


## 2. Download public Perez data

The notebook records URLs, sizes, and hashes. Existing validated files are reused.

In [ ]:
URLS = {
    'counts': (
        'https://ftp.ncbi.nlm.nih.gov/geo/series/'
        'GSE157nnn/GSE157204/suppl/'
        'GSE157204_Processed_count_data.tsv.gz'
    ),
    'sample_metadata': (
        'https://cdn.elifesciences.org/articles/63092/'
        'elife-63092-supp1-v2.tsv'
    ),
    'published_de': (
        'https://cdn.elifesciences.org/articles/63092/'
        'elife-63092-supp5-v2.tsv'
    ),
}
PATHS = {
    'counts': DIRS['raw'] / 'GSE157204_Processed_count_data.tsv.gz',
    'sample_metadata': DIRS['raw'] / 'elife-63092-supp1-v2.tsv',
    'published_de': DIRS['raw'] / 'elife-63092-supp5-v2.tsv',
}

def download_stream(url, destination, chunk_size=1024*1024):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print('Reusing', destination.name)
        return
    temporary = destination.with_name(destination.name + '.part')
    with requests.get(url, stream=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        with open(temporary, 'wb') as handle:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    handle.write(chunk)
    os.replace(temporary, destination)
    print('Downloaded', destination.name, f'{destination.stat().st_size/1e6:.2f} MB')

def sha256_file(path, chunk_size=1024*1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

for key in URLS:
    download_stream(URLS[key], PATHS[key])

download_manifest = {
    key: {
        'url': URLS[key],
        'bytes': int(PATHS[key].stat().st_size),
        'sha256': sha256_file(PATHS[key]),
    }
    for key in URLS
}
with open(DIRS['manifests'] / 'download_manifest.json', 'w') as handle:
    json.dump(download_manifest, handle, indent=2)

counts_raw = pd.read_csv(PATHS['counts'], sep='\t', compression='gzip', low_memory=False)
sample_metadata = pd.read_csv(PATHS['sample_metadata'], sep='\t', low_memory=False)
published_de = pd.read_csv(PATHS['published_de'], sep='\t', low_memory=False)

print('Counts:', counts_raw.shape)
print('Metadata:', sample_metadata.shape)
print('Published DE:', published_de.shape)

## 3. Validate samples and freeze published candidate labels

Published `Dendrites` and `Somata` classifications are used. `Constant` genes, ERCC controls, and mitochondrial-genome symbols are excluded. No G/rG4 feature is used here.

In [ ]:
GENE_COLUMNS = ['GeneID', 'Gene_name', 'Coordinates', 'Length']
sample_columns = [column for column in counts_raw.columns if column not in GENE_COLUMNS]
count_samples = set(sample_columns)
metadata_samples = set(sample_metadata['Sample_ID'].astype(str))

assert len(sample_columns) == 960
assert sample_metadata['Sample_ID'].nunique() == 960
assert count_samples == metadata_samples

sample_qc = sample_metadata.groupby('Comparment').agg(
    n_samples=('Sample_ID', 'size'),
    mean_features=('nFeature_RNA', 'mean'),
    median_features=('nFeature_RNA', 'median'),
    mean_counts=('nCount_RNA', 'mean'),
    median_counts=('nCount_RNA', 'median'),
).reset_index()
sample_qc.to_csv(DIRS['data'] / 'perez_sample_qc.csv', index=False, float_format=FLOAT_FORMAT)
display(sample_qc)

de = published_de.copy()
for column in ['SomataNormExp', 'DendritesNormExp', 'p-value', 'Avg_logFC', 'pct.Den', 'pct.Som', 'adj_p-value', 'CellNormExp']:
    de[column] = pd.to_numeric(de[column], errors='coerce')
de['Gene'] = de['Gene'].astype(str).str.strip()
de['Evaluation'] = de['Evaluation'].astype(str).str.strip()

de = de[
    de['Gene'].notna()
    & de['adj_p-value'].notna()
    & de['CellNormExp'].notna()
    & ~de['Gene'].str.startswith('ERCC-', na=False)
    & ~de['Gene'].str.match(r'^(mt-|Mt-|MT-)', na=False)
].copy()

dendritic_candidates = de[
    (de['adj_p-value'] < 0.05)
    & (de['Evaluation'] == 'Dendrites')
    & (de['Avg_logFC'] > 0)
].copy()
soma_candidates = de[
    (de['adj_p-value'] < 0.05)
    & (de['Evaluation'] == 'Somata')
    & (de['Avg_logFC'] < 0)
].copy()

dendritic_candidates['published_group'] = 'dendritic_enriched'
soma_candidates['published_group'] = 'soma_enriched'
presequence_panel = pd.concat([dendritic_candidates, soma_candidates], ignore_index=True)
presequence_panel['dataset'] = 'Perez_2021_GSE157204'
presequence_panel['species'] = 'Rattus_norvegicus'
presequence_panel['selection_source'] = 'published_supplementary_file_5'

assert presequence_panel['Gene'].is_unique
assert (presequence_panel['published_group'] == 'dendritic_enriched').sum() == 406
assert (presequence_panel['published_group'] == 'soma_enriched').sum() == 696
presequence_panel.to_csv(DIRS['panels'] / 'perez_published_candidate_panel.csv', index=False, float_format=FLOAT_FORMAT)

print(presequence_panel['published_group'].value_counts())
print('Compound/readthrough symbols:', presequence_panel['Gene'].str.contains('-', regex=False).sum())

## 4. Download fixed rat annotation

Ensembl release 116 and GRCr8 are pinned. Both canonical-prioritized and longest eligible 3′UTRs are exported.

In [ ]:
ENSEMBL_RELEASE = 116
RAT_ASSEMBLY = 'GRCr8'
ENSEMBL_BASE = f'https://ftp.ensembl.org/pub/release-{ENSEMBL_RELEASE}'
ANNOTATION_URLS = {
    'gtf': f'{ENSEMBL_BASE}/gtf/rattus_norvegicus/Rattus_norvegicus.{RAT_ASSEMBLY}.{ENSEMBL_RELEASE}.gtf.gz',
    'cdna': f'{ENSEMBL_BASE}/fasta/rattus_norvegicus/cdna/Rattus_norvegicus.{RAT_ASSEMBLY}.cdna.all.fa.gz',
    'cds': f'{ENSEMBL_BASE}/fasta/rattus_norvegicus/cds/Rattus_norvegicus.{RAT_ASSEMBLY}.cds.all.fa.gz',
}
ANNOTATION_PATHS = {
    'gtf': DIRS['annotation'] / f'Rattus_norvegicus.{RAT_ASSEMBLY}.{ENSEMBL_RELEASE}.gtf.gz',
    'cdna': DIRS['annotation'] / f'Rattus_norvegicus.{RAT_ASSEMBLY}.cdna.all.fa.gz',
    'cds': DIRS['annotation'] / f'Rattus_norvegicus.{RAT_ASSEMBLY}.cds.all.fa.gz',
}

for key in ANNOTATION_URLS:
    download_stream(ANNOTATION_URLS[key], ANNOTATION_PATHS[key])
    with gzip.open(ANNOTATION_PATHS[key], 'rt') as handle:
        first = next(line.strip() for line in handle if line.strip())
    expected = '#' if key == 'gtf' else '>'
    assert first.startswith(expected)

annotation_manifest = {
    'release': ENSEMBL_RELEASE,
    'assembly': RAT_ASSEMBLY,
    'resources': {
        key: {
            'url': ANNOTATION_URLS[key],
            'bytes': int(ANNOTATION_PATHS[key].stat().st_size),
            'sha256': sha256_file(ANNOTATION_PATHS[key]),
        }
        for key in ANNOTATION_URLS
    },
}
with open(DIRS['manifests'] / 'annotation_manifest.json', 'w') as handle:
    json.dump(annotation_manifest, handle, indent=2)
print(json.dumps(annotation_manifest, indent=2))

## 5. Parse transcripts and extract 3′UTRs

In [ ]:
def parse_gtf_attributes(text):
    output = defaultdict(list)
    for key, value in re.findall(r'(\S+)\s+"([^"]*)"', text):
        output[key].append(value)
    return dict(output)

def first_attr(attributes, *names, default=None):
    for name in names:
        values = attributes.get(name, [])
        if values:
            return values[0]
    return default

def strip_version(identifier):
    return str(identifier).split('.')[0]

def parse_tsl(value):
    match = re.search(r'\d+', str(value)) if value is not None else None
    return int(match.group()) if match else 99

def clean_sequence(sequence):
    return re.sub(r'[^ACGTN]', '', str(sequence).upper().replace('U', 'T'))

candidate_genes = set(presequence_panel['Gene'])
records = []
with gzip.open(ANNOTATION_PATHS['gtf'], 'rt') as handle:
    for line in handle:
        if not line or line.startswith('#'):
            continue
        fields = line.rstrip('\n').split('\t')
        if len(fields) != 9 or fields[2] != 'transcript':
            continue
        attrs = parse_gtf_attributes(fields[8])
        gene = first_attr(attrs, 'gene_name')
        if gene not in candidate_genes:
            continue
        gene_type = first_attr(attrs, 'gene_biotype', 'gene_type')
        tx_type = first_attr(attrs, 'transcript_biotype', 'transcript_type')
        if gene_type != 'protein_coding' or tx_type != 'protein_coding':
            continue
        tx_versioned = first_attr(attrs, 'transcript_id')
        if tx_versioned is None:
            continue
        tags = attrs.get('tag', [])
        records.append({
            'Gene': gene,
            'gene_id': strip_version(first_attr(attrs, 'gene_id')),
            'transcript_id': strip_version(tx_versioned),
            'transcript_id_versioned': tx_versioned,
            'transcript_name': first_attr(attrs, 'transcript_name'),
            'gene_biotype': gene_type,
            'transcript_biotype': tx_type,
            'transcript_support_level': parse_tsl(first_attr(attrs, 'transcript_support_level')),
            'level': int(first_attr(attrs, 'level', default=99)),
            'tags': ';'.join(sorted(set(tags))),
            'is_ensembl_canonical': 'Ensembl_canonical' in tags,
            'is_mane_select': 'MANE_Select' in tags,
            'is_appris_principal': any(str(tag).startswith('appris_principal') for tag in tags),
            'is_ccds': 'CCDS' in tags,
        })

transcripts = pd.DataFrame(records).drop_duplicates(['Gene', 'transcript_id']).reset_index(drop=True)
if transcripts.empty:
    raise RuntimeError('No protein-coding candidate transcripts found.')
target_ids = set(transcripts['transcript_id'])


def read_target_fasta(path, target_ids):
    sequences, current_id, parts = {}, None, []
    def save():
        if current_id in target_ids:
            sequences[current_id] = clean_sequence(''.join(parts))
    with gzip.open(path, 'rt') as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith('>'):
                save()
                current_id = strip_version(line[1:].split()[0])
                parts = []
            elif current_id in target_ids:
                parts.append(line)
        save()
    return sequences

cdna_sequences = read_target_fasta(ANNOTATION_PATHS['cdna'], target_ids)
cds_sequences = read_target_fasta(ANNOTATION_PATHS['cds'], target_ids)
assert set(target_ids) == set(cdna_sequences) == set(cds_sequences)


def extract_utr3(cdna, cds):
    cdna, cds = clean_sequence(cdna), clean_sequence(cds)
    if not cdna: return '', 'missing_cdna'
    if not cds: return '', 'missing_cds'
    first = cdna.find(cds)
    if first < 0: return '', 'cds_not_found_in_cdna'
    if cdna.find(cds, first + 1) >= 0: return '', 'nonunique_cds_match'
    utr = cdna[first + len(cds):]
    if len(utr) == 0: return '', 'empty_utr3'
    if len(utr) < MIN_UTR_LENGTH: return utr, 'short_utr3'
    return utr, 'eligible'

extractions = []
for row in transcripts.itertuples(index=False):
    utr, status = extract_utr3(cdna_sequences[row.transcript_id], cds_sequences[row.transcript_id])
    extractions.append({
        'Gene': row.Gene,
        'transcript_id': row.transcript_id,
        'cdna_length': len(cdna_sequences[row.transcript_id]),
        'cds_length': len(cds_sequences[row.transcript_id]),
        'utr3_length': len(utr),
        'utr3_status': status,
        'utr3_sequence': utr,
        'utr3_sha256': hashlib.sha256(utr.encode()).hexdigest() if utr else '',
    })
transcripts = transcripts.merge(pd.DataFrame(extractions), on=['Gene', 'transcript_id'], validate='one_to_one')
transcripts.drop(columns='utr3_sequence').to_csv(
    DIRS['annotation'] / 'all_candidate_transcript_extraction_metadata.csv', index=False, float_format=FLOAT_FORMAT
)
print(transcripts['utr3_status'].value_counts())

## 6. Select representative and longest 3′UTRs

Selection uses annotation evidence only. Sequence composition is not calculated until after matching.

In [ ]:
eligible = transcripts[transcripts['utr3_status'] == 'eligible'].copy()
eligible = eligible.sort_values(
    ['Gene', 'is_ensembl_canonical', 'is_mane_select', 'is_appris_principal', 'is_ccds',
     'transcript_support_level', 'level', 'cdna_length', 'transcript_id'],
    ascending=[True, False, False, False, False, True, True, False, True]
)
canonical_utr = eligible.drop_duplicates('Gene', keep='first').copy()
canonical_utr['sequence_definition'] = 'annotation_prioritized'
longest_utr = eligible.sort_values(
    ['Gene', 'utr3_length', 'is_ensembl_canonical', 'transcript_id'],
    ascending=[True, False, False, True]
).drop_duplicates('Gene', keep='first').copy()
longest_utr['sequence_definition'] = 'longest_eligible_utr3'

localization_columns = ['Gene', 'published_group', 'SomataNormExp', 'DendritesNormExp',
                        'CellNormExp', 'Avg_logFC', 'pct.Den', 'pct.Som', 'adj_p-value']
localization = presequence_panel[localization_columns].drop_duplicates('Gene')
canonical_utr = localization.merge(canonical_utr, on='Gene', validate='one_to_one')
longest_utr = localization.merge(longest_utr, on='Gene', validate='one_to_one')

for name, table in [('canonical', canonical_utr), ('longest', longest_utr)]:
    table.drop(columns='utr3_sequence').to_csv(
        DIRS['annotation'] / f'{name}_3utr_metadata.csv', index=False, float_format=FLOAT_FORMAT
    )
    with open(DIRS['annotation'] / f'{name}_3utr.fa', 'w') as handle:
        for row in table.itertuples(index=False):
            handle.write(f'>{row.Gene}|{row.transcript_id}|{row.sequence_definition}\n')
            for start in range(0, len(row.utr3_sequence), 80):
                handle.write(row.utr3_sequence[start:start+80] + '\n')

represented = set(canonical_utr['Gene'])
status = presequence_panel[['Gene', 'published_group', 'Avg_logFC', 'CellNormExp']].copy()
status['represented_in_protein_coding_gtf'] = status['Gene'].isin(set(transcripts['Gene']))
status['eligible_utr3'] = status['Gene'].isin(represented)
status['annotation_status'] = np.select(
    [~status['represented_in_protein_coding_gtf'],
     status['represented_in_protein_coding_gtf'] & ~status['eligible_utr3'],
     status['eligible_utr3']],
    ['not_mapped_to_protein_coding_gtf', 'mapped_but_no_eligible_utr3', 'eligible_utr3'],
    default='unexpected'
)
status.to_csv(DIRS['panels'] / 'candidate_annotation_status.csv', index=False)
display(pd.crosstab(status['published_group'], status['annotation_status']))

## 7. Sequence-blind joint matching and panel freeze

A joint mixed-integer optimization selects 250 genes per group from common support. The smallest prespecified feasible mean-balance tolerance is used, followed by optimal one-to-one pairing.

In [ ]:
analysis_candidates = canonical_utr.copy()
analysis_candidates['match_expression'] = np.log1p(analysis_candidates['CellNormExp'].astype(float))
analysis_candidates['match_utr_length'] = np.log(analysis_candidates['utr3_length'].astype(float))
dendritic = analysis_candidates[analysis_candidates['published_group'] == 'dendritic_enriched'].copy().reset_index(drop=True)
soma = analysis_candidates[analysis_candidates['published_group'] == 'soma_enriched'].copy().reset_index(drop=True)
assert len(dendritic) >= N_MATCHED_PER_GROUP and len(soma) >= N_MATCHED_PER_GROUP

MATCH_VARIABLES = ['match_expression', 'match_utr_length']
combined = pd.concat([dendritic[MATCH_VARIABLES], soma[MATCH_VARIABLES]], ignore_index=True)
means, sds = combined.mean(), combined.std(ddof=1)
for variable in MATCH_VARIABLES:
    dendritic[variable + '_z'] = (dendritic[variable] - means[variable]) / sds[variable]
    soma[variable + '_z'] = (soma[variable] - means[variable]) / sds[variable]
MATCH_Z = [name + '_z' for name in MATCH_VARIABLES]
D = dendritic[MATCH_Z].to_numpy(float)
S = soma[MATCH_Z].to_numpy(float)
n_d, n_s = len(D), len(S)

cost = np.concatenate([
    ((D[:, None, :] - S[None, :, :])**2).sum(2).min(1),
    ((S[:, None, :] - D[None, :, :])**2).sum(2).min(1),
]) + 1e-10 * np.arange(n_d + n_s)


def solve_selection(tolerance):
    rows = [np.r_[np.ones(n_d), np.zeros(n_s)], np.r_[np.zeros(n_d), np.ones(n_s)]]
    lower = [N_MATCHED_PER_GROUP, N_MATCHED_PER_GROUP]
    upper = [N_MATCHED_PER_GROUP, N_MATCHED_PER_GROUP]
    for axis in range(len(MATCH_Z)):
        rows.append(np.r_[D[:, axis], -S[:, axis]])
        lower.append(-N_MATCHED_PER_GROUP * tolerance)
        upper.append(+N_MATCHED_PER_GROUP * tolerance)
    return milp(
        c=cost,
        integrality=np.ones(n_d + n_s, dtype=int),
        bounds=Bounds(np.zeros(n_d + n_s), np.ones(n_d + n_s)),
        constraints=LinearConstraint(np.vstack(rows), np.asarray(lower), np.asarray(upper)),
        options={'time_limit': 300},
    )

TOLERANCE_GRID = [0.025, 0.05, 0.075, 0.10]
selection, tolerance_used = None, None
for tolerance in TOLERANCE_GRID:
    candidate = solve_selection(tolerance)
    print('Tolerance', tolerance, candidate.message)
    if candidate.success:
        selection, tolerance_used = candidate, tolerance
        break
if selection is None:
    raise RuntimeError('No feasible matched design.')

chosen = selection.x > 0.5
selected_d = dendritic.iloc[np.flatnonzero(chosen[:n_d])].copy().reset_index(drop=True)
selected_s = soma.iloc[np.flatnonzero(chosen[n_d:])].copy().reset_index(drop=True)
assert len(selected_d) == len(selected_s) == N_MATCHED_PER_GROUP

pair_cost = ((selected_d[MATCH_Z].to_numpy()[:, None, :] - selected_s[MATCH_Z].to_numpy()[None, :, :])**2).sum(2)
d_order, s_order = linear_sum_assignment(pair_cost)
final_dendritic = selected_d.iloc[d_order].copy().reset_index(drop=True)
final_soma = selected_s.iloc[s_order].copy().reset_index(drop=True)
final_dendritic['matched_soma_gene'] = final_soma['Gene'].to_numpy()
final_soma['matched_dendritic_gene'] = final_dendritic['Gene'].to_numpy()
final_soma['matching_distance'] = np.sqrt(pair_cost[d_order, s_order])
final_dendritic['analysis_group'] = 'dendritic_enriched'
final_soma['analysis_group'] = 'soma_enriched_matched'
final_panel = pd.concat([final_dendritic, final_soma], ignore_index=True)
assert len(final_panel) == 500 and final_panel['Gene'].is_unique

pair_diagnostics = pd.DataFrame({
    'dendritic_gene': final_dendritic['Gene'],
    'soma_gene': final_soma['Gene'],
    'matching_distance': final_soma['matching_distance'],
    'dendritic_CellNormExp': final_dendritic['CellNormExp'],
    'soma_CellNormExp': final_soma['CellNormExp'],
    'dendritic_utr3_length': final_dendritic['utr3_length'],
    'soma_utr3_length': final_soma['utr3_length'],
})

def smd(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    pooled = np.sqrt((np.var(a, ddof=1) + np.var(b, ddof=1))/2)
    return float((np.mean(a)-np.mean(b))/pooled)

balance = pd.DataFrame([
    {'variable': variable,
     'dendritic_mean': float(final_dendritic[variable].mean()),
     'soma_mean': float(final_soma[variable].mean()),
     'standardized_mean_difference': smd(final_dendritic[variable], final_soma[variable]),
     'paired_spearman_rho': float(stats.spearmanr(final_dendritic[variable], final_soma[variable]).statistic)}
    for variable in MATCH_VARIABLES
])
if (balance['standardized_mean_difference'].abs() > 0.1).any():
    raise RuntimeError('Matching failed |SMD| < 0.1 criterion.')

trim_cutoff = pair_diagnostics['matching_distance'].quantile(0.95)
retained_pairs = pair_diagnostics[pair_diagnostics['matching_distance'] <= trim_cutoff].copy()
trimmed_genes = set(retained_pairs['dendritic_gene']) | set(retained_pairs['soma_gene'])
trimmed_panel = final_panel[final_panel['Gene'].isin(trimmed_genes)].copy()

final_panel.to_csv(DIRS['panels'] / 'perez_frozen_matched_panel.csv', index=False, float_format=FLOAT_FORMAT)
pair_diagnostics.to_csv(DIRS['panels'] / 'perez_pair_diagnostics.csv', index=False, float_format=FLOAT_FORMAT)
balance.to_csv(DIRS['panels'] / 'perez_matching_balance.csv', index=False, float_format=FLOAT_FORMAT)
trimmed_panel.to_csv(DIRS['panels'] / 'perez_frozen_panel_trim95.csv', index=False, float_format=FLOAT_FORMAT)
retained_pairs.to_csv(DIRS['panels'] / 'perez_retained_pairs_trim95.csv', index=False, float_format=FLOAT_FORMAT)

matching_manifest = {
    'tolerance_grid': TOLERANCE_GRID,
    'smallest_feasible_tolerance': tolerance_used,
    'n_per_group': N_MATCHED_PER_GROUP,
    'matching_variables': MATCH_VARIABLES,
    'sequence_features_used': False,
    'trim95_cutoff': float(trim_cutoff),
    'trim95_pairs': int(len(retained_pairs)),
}
with open(DIRS['manifests'] / 'matching_manifest.json', 'w') as handle:
    json.dump(matching_manifest, handle, indent=2)

print(final_panel['analysis_group'].value_counts())
display(balance)
print('Trimmed pairs:', len(retained_pairs))

### 7b. Panel hash and support diagnostics

In [ ]:
# Panel hash and support diagnostics (harmonised with the Tushev workflow).
PANEL_HASH = hashlib.sha256(
    pd.util.hash_pandas_object(
        final_panel[['Gene', 'analysis_group', 'utr3_length']], index=False
    ).values.tobytes()
).hexdigest()

support = pd.DataFrame([{
    'dataset': DATASET_ID,
    'group': group,
    'candidates_published': int((presequence_panel['published_group'] == group).sum()),
    'eligible_utr3_pool': int(pool_size),
    'selected': N_MATCHED_PER_GROUP,
    'support_exhaustion': N_MATCHED_PER_GROUP / pool_size,
} for group, pool_size in [('dendritic_enriched', len(dendritic)),
                           ('soma_enriched', len(soma))]])
support['matching_tolerance_used'] = tolerance_used
support['trim95_cutoff'] = float(trim_cutoff)
support['trim95_pairs'] = int(len(retained_pairs))
support['panel_hash'] = PANEL_HASH
support.to_csv(DIRS['panels'] / 'perez_support_diagnostics.csv',
               index=False, float_format=FLOAT_FORMAT)

print('Panel hash:', PANEL_HASH)
display(support)


## 8. Calculate sequence features and frozen scores

In [ ]:
def merged_g4_density(sequence, minimum_g_run=3, maximum_loop=7):
    sequence = clean_sequence(sequence)
    pattern = re.compile(
        rf'G{{{minimum_g_run},}}[ACGTN]{{1,{maximum_loop}}}'
        rf'G{{{minimum_g_run},}}[ACGTN]{{1,{maximum_loop}}}'
        rf'G{{{minimum_g_run},}}[ACGTN]{{1,{maximum_loop}}}'
        rf'G{{{minimum_g_run},}}'
    )
    spans = [match.span() for match in pattern.finditer(sequence)]
    if not spans:
        return 0.0
    spans.sort(); merged = [list(spans[0])]
    for start, end in spans[1:]:
        if start <= merged[-1][1]: merged[-1][1] = max(merged[-1][1], end)
        else: merged.append([start, end])
    return len(merged) * 1000.0 / len(sequence)


def sequence_features(sequence):
    sequence = clean_sequence(sequence)
    length = len(sequence)
    if length < MIN_UTR_LENGTH:
        raise ValueError('Short sequence entered frozen panel.')
    return {
        'utr3_length': length,
        'g_frac': sequence.count('G')/length,
        'gc_fraction': (sequence.count('G')+sequence.count('C'))/length,
        'rg4_density': merged_g4_density(sequence, 3, 7),
        'rg4_relaxed_density': merged_g4_density(sequence, 2, 12),
    }

sequence_map = canonical_utr.set_index('Gene')['utr3_sequence'].to_dict()
features = pd.DataFrame({gene: sequence_features(sequence_map[gene]) for gene in final_panel['Gene']}).T
features.index.name = 'Gene'
features = features.join(final_panel.set_index('Gene')[['analysis_group', 'published_group', 'CellNormExp', 'Avg_logFC']])
assert len(features) == 500 and not features.isna().any().any()


def zscore(values):
    values = pd.Series(values, index=features.index, dtype=float)
    sd = values.std(ddof=1)
    if not np.isfinite(sd) or sd == 0:
        raise ValueError(f'Cannot standardize {values.name}')
    return (values-values.mean())/sd

scores = pd.DataFrame(index=features.index)
scores['g_only'] = zscore(features['g_frac'])
scores['rg4_only'] = zscore(features['rg4_density'])
scores['primary_g_rg4'] = zscore((zscore(features['g_frac']) + zscore(features['rg4_density']))/2)
X = sm.add_constant(pd.DataFrame({
    'g_frac': zscore(features['g_frac']),
    'log_utr_length': zscore(np.log(features['utr3_length'])),
}, index=features.index))
rg4_model = sm.OLS(zscore(features['rg4_density']), X).fit()
scores['rg4_residual'] = zscore(rg4_model.resid)
scores['gc_control'] = zscore(features['gc_fraction'])
scores['length_control'] = zscore(np.log(features['utr3_length']))
scores['relaxed_rg4'] = zscore(features['rg4_relaxed_density'])
rng = np.random.default_rng(PANEL_SEED)
scores['shuffled_primary'] = rng.permutation(scores['primary_g_rg4'].to_numpy())
scores['random_score'] = rng.normal(size=len(scores))
scores = scores.join(features[['analysis_group', 'published_group', 'CellNormExp', 'Avg_logFC', 'utr3_length', 'g_frac', 'rg4_density']])

features.to_csv(DIRS['scores'] / 'perez_sequence_features.csv', float_format=FLOAT_FORMAT)
scores.to_csv(DIRS['scores'] / 'perez_sequence_scores.csv', float_format=FLOAT_FORMAT)

print('Features:', features.shape)
print('Zero strict rG4:', (features['rg4_density']==0).mean())
print('G-rG4 Spearman:', stats.spearmanr(features['g_frac'], features['rg4_density']))

### 8b. Sequence-feature sparsity diagnostics

In [ ]:
# Sequence-feature sparsity. Strict rG4 sites are rare here, so primary_g_rg4 is
# nearly collinear with g_only. This is reported as a dataset property, not corrected.
sparsity_rows = []
for column, label in [('rg4_density', 'strict_rg4'),
                      ('rg4_relaxed_density', 'relaxed_rg4')]:
    values = features[column].astype(float)
    sparsity_rows.append({
        'dataset': DATASET_ID, 'feature': label, 'n': int(len(values)),
        'zero_fraction': float((values == 0).mean()),
        'median': float(values.median()), 'mean': float(values.mean()),
        'sd': float(values.std(ddof=1)),
    })

g_rg4 = stats.spearmanr(features['g_frac'].astype(float),
                        features['rg4_density'].astype(float))
redundancy = stats.spearmanr(scores['primary_g_rg4'].astype(float),
                             scores['g_only'].astype(float))

sparsity = pd.DataFrame(sparsity_rows)
sparsity['g_vs_rg4_spearman_rho'] = float(g_rg4.statistic)
sparsity['g_vs_rg4_spearman_p'] = float(g_rg4.pvalue)
sparsity['primary_vs_gonly_spearman_rho'] = float(redundancy.statistic)
sparsity['panel_hash'] = PANEL_HASH

sparsity.to_csv(DIRS['statistics'] / 'perez_sequence_sparsity.csv',
                index=False, float_format=FLOAT_FORMAT)
sparsity.to_csv(DIRS['plotting'] / 'panel_B_perez_sequence_sparsity.csv',
                index=False, float_format=FLOAT_FORMAT)
display(sparsity)


## 9. Sequence-level paired statistics

In [ ]:
def inference(values, n_boot=N_BOOT, n_permutations=N_PERMUTATIONS, seed=20260831):
    values = np.asarray(values, dtype=float)
    if len(values) < 5 or not np.isfinite(values).all():
        raise ValueError('Invalid inference vector.')
    rng = np.random.default_rng(seed)
    observed = float(values.mean())
    boot_idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    boot = values[boot_idx].mean(axis=1)
    signs = rng.choice(np.array([-1.0, 1.0]), size=(n_permutations, len(values)))
    perm = (signs*values).mean(axis=1)
    p = (1+np.sum(np.abs(perm)>=abs(observed)))/(n_permutations+1)
    return {
        'n': int(len(values)), 'estimate': observed, 'median': float(np.median(values)),
        'sd': float(values.std(ddof=1)), 'ci_low': float(np.percentile(boot,2.5)),
        'ci_high': float(np.percentile(boot,97.5)), 'empirical_p': float(p),
        'proportion_positive': float((values>0).mean()),
    }

score_names = ['primary_g_rg4','g_only','rg4_only','rg4_residual','gc_control','length_control','relaxed_rg4','shuffled_primary','random_score']
sequence_rows=[]
for score_name in score_names:
    d = scores.loc[pair_diagnostics['dendritic_gene'], score_name].to_numpy(float)
    s = scores.loc[pair_diagnostics['soma_gene'], score_name].to_numpy(float)
    result = inference(d-s, seed=20260900+len(sequence_rows))
    mw = stats.mannwhitneyu(d,s,alternative='two-sided')
    result.update({
        'dataset':'Perez_2021_GSE157204','score':score_name,
        'mean_dendritic':float(d.mean()),'mean_soma':float(s.mean()),
        'cliffs_delta':float(2*mw.statistic/(len(d)*len(s))-1),
        'mann_whitney_p':float(mw.pvalue),
    })
    sequence_rows.append(result)
sequence_effects = pd.DataFrame(sequence_rows)
sequence_effects['q_bh_empirical'] = multipletests(sequence_effects['empirical_p'], method='fdr_bh')[1]
sequence_effects.to_csv(DIRS['statistics'] / 'perez_sequence_effects.csv', index=False, float_format=FLOAT_FORMAT)
sequence_effects.to_csv(DIRS['plotting'] / 'panel_B_perez_sequence_effects.csv', index=False, float_format=FLOAT_FORMAT)
display(sequence_effects.sort_values('empirical_p'))

### 9b. Trim-95 sensitivity for paired sequence effects

In [ ]:
# Trim-95 sensitivity: identical paired inference on the 95 percent best-matched pairs.
trim_rows = []
for score_name in score_names:
    d = scores.loc[retained_pairs['dendritic_gene'], score_name].to_numpy(float)
    s = scores.loc[retained_pairs['soma_gene'], score_name].to_numpy(float)
    result = inference(d - s, seed=20260950 + len(trim_rows))
    mw = stats.mannwhitneyu(d, s, alternative='two-sided')
    result.update({
        'dataset': DATASET_ID, 'score': score_name, 'analysis': 'trim95_sensitivity',
        'mean_dendritic': float(d.mean()), 'mean_soma': float(s.mean()),
        'cliffs_delta': float(2 * mw.statistic / (len(d) * len(s)) - 1),
        'mann_whitney_p': float(mw.pvalue), 'panel_hash': PANEL_HASH,
    })
    trim_rows.append(result)

sequence_effects_trim95 = pd.DataFrame(trim_rows)
sequence_effects_trim95['q_bh_empirical'] = multipletests(
    sequence_effects_trim95['empirical_p'], method='fdr_bh')[1]
sequence_effects_trim95.to_csv(DIRS['statistics'] / 'perez_sequence_effects_trim95.csv',
                               index=False, float_format=FLOAT_FORMAT)
sequence_effects_trim95.to_csv(DIRS['plotting'] / 'panel_B_perez_sequence_effects_trim95.csv',
                               index=False, float_format=FLOAT_FORMAT)

concordance = (
    sequence_effects[['score', 'estimate', 'empirical_p']]
    .rename(columns={'estimate': 'estimate_full', 'empirical_p': 'p_full'})
    .merge(sequence_effects_trim95[['score', 'estimate', 'empirical_p']]
           .rename(columns={'estimate': 'estimate_trim95', 'empirical_p': 'p_trim95'}),
           on='score')
)
concordance['sign_agreement'] = (
    np.sign(concordance['estimate_full']) == np.sign(concordance['estimate_trim95'])
)
concordance.to_csv(DIRS['statistics'] / 'perez_trim95_concordance.csv',
                   index=False, float_format=FLOAT_FORMAT)
display(concordance)


## 10. Upload and freeze the MSC model

In [ ]:
# Model and config were already uploaded and verified in section 1.
assert MODEL_PATH.exists() and CFG_PATH.exists()

if str(MODEL_PATH.parent) not in sys.path:
    sys.path.insert(0, str(MODEL_PATH.parent))

import msc_model_feedback as M
importlib.reload(M)
from msc_model_feedback import simulate_one, sample_params, load_config

CFG = dict(load_config(str(CFG_PATH)))
ORIGINAL_MATERIAL_POOL = M.make_material_pool
assert MODEL_VARIANT in M.MODEL_VARIANTS

shutil.copy2(MODEL_PATH, DIRS['model'] / 'msc_model_feedback_used.py')
shutil.copy2(CFG_PATH, DIRS['model'] / 'config_msc_feedback_used.yaml')

M.equation_mapping().to_csv(DIRS['model'] / 'equation_to_code_mapping.csv',
                            index=False, float_format=FLOAT_FORMAT)

with open(DIRS['manifests'] / 'model_manifest.json', 'w') as handle:
    json.dump({'model_sha256': sha256_file(MODEL_PATH),
               'config_sha256': sha256_file(CFG_PATH),
               'model_variant': MODEL_VARIANT,
               'config_keys': list(CFG)}, handle, indent=2)

print('Model and config loaded. Variant:', MODEL_VARIANT)


## 11. Lock simulation plan and PSD calibration

In [ ]:
JIA = {
    'E_modulus_kPa': {'ME':4.7,'WT':17.4},
    'tau2_ms': {'ME':4.8,'WT':57.0},
    'mobile_fraction': {'ME':49.1,'WT':16.0},
}
def oriented_log_fc(name):
    values=JIA[name]
    return np.log(values['ME']/values['WT']) if name=='mobile_fraction' else np.log(values['WT']/values['ME'])
JIA_RAW={name:oriented_log_fc(name) for name in JIA}
JIA_MEAN=float(np.mean(list(JIA_RAW.values())))
PSD_GAIN=0.55
PSD_SEPARATION={name:PSD_GAIN*value/JIA_MEAN for name,value in JIA_RAW.items()}
ANALYSES={
    'primary_g_rg4':{'score':'primary_g_rg4','mode':'correct'},
    'g_only':{'score':'g_only','mode':'correct'},
    'primary_mapping_shuffled':{'score':'primary_g_rg4','mode':'shuffled'},
}
OBSERVABLES={'tau2_ms':PRIMARY_N_SEEDS,'mobile_fraction':SENSITIVITY_N_SEEDS,'E_modulus_kPa':SENSITIVITY_N_SEEDS}
RUN_PLAN=[]
for observable,n_seeds in OBSERVABLES.items():
    for analysis_name,spec in ANALYSES.items():
        for state,sign in [('WT_like',1.0),('ME_like',-1.0)]:
            RUN_PLAN.append({'observable':observable,'n_seeds':n_seeds,'analysis_name':analysis_name,
                             'score_definition':spec['score'],'mode':spec['mode'],'psd_state':state,
                             'anchor':float(sign*PSD_SEPARATION[observable])})
plan_manifest={'created_at':datetime.datetime.now().isoformat(),'psd_gain':PSD_GAIN,
               'psd_separation':PSD_SEPARATION,'observables':OBSERVABLES,'analyses':ANALYSES,
               'total_simulations':int(sum(row['n_seeds'] for row in RUN_PLAN)),
               'parameter_seed_base':PARAMETER_SEED_BASE,'simulation_seed_base':SIMULATION_SEED_BASE}
with open(DIRS['manifests']/'simulation_plan.json','w') as handle: json.dump(plan_manifest,handle,indent=2)
print(json.dumps(plan_manifest,indent=2))

## 12. Injection, one-seed runner, and compact transcript counts

In [ ]:
LABELS=final_panel['Gene'].tolist()
GROUP_MAP=final_panel.set_index('Gene')['analysis_group'].to_dict()
COORDINATES={name:scores.loc[LABELS,name].to_numpy(float) for name in ['primary_g_rg4','g_only']}

def pvalue(parameters,names,default=None):
    for name in names:
        if name in parameters:return parameters[name]
    if default is not None:return default
    raise KeyError(names)

def make_pool(
    coordinates,
    anchor,
    mode
):
    shifted = (
        np.asarray(
            coordinates,
            dtype=float
        )
        - float(anchor)
    )

    def pool(parameters, rng):
        D = int(
            pvalue(
                parameters,
                ["D_material", "D"],
                1
            )
        )

        N = int(
            pvalue(
                parameters,
                ["Ngran", "N_gran"]
            )
        )

        ell = float(
            pvalue(
                parameters,
                ["cargo_ell"],
                1.0
            )
        )

        strong_fraction = float(
            pvalue(
                parameters,
                ["strong_fraction"],
                0.3
            )
        )

        transcript_indices = rng.choice(
            len(shifted),
            size=N,
            replace=(N > len(shifted))
        )

        true_scalar = shifted[
            transcript_indices
        ].copy()

        if mode == "correct":
            used_scalar = (
                true_scalar.copy()
            )

        elif mode == "shuffled":
            used_scalar = true_scalar[
                rng.permutation(N)
            ].copy()

        elif mode == "random":
            used_scalar = rng.normal(
                0,
                1.25,
                size=N
            )

        else:
            raise ValueError(
                f"Unknown mode: {mode}"
            )

        # The remaining material dimensions are shared
        # between authentic and model-used representations.
        shared_dimensions = np.zeros(
            (N, D),
            dtype=float
        )

        if D > 1:
            shared_dimensions[:, 1:] = (
                rng.normal(
                    0,
                    0.3,
                    size=(N, D - 1)
                )
            )

        authentic_material = (
            shared_dimensions.copy()
        )

        model_used_material = (
            shared_dimensions.copy()
        )

        authentic_material[:, 0] = (
            true_scalar
        )

        model_used_material[:, 0] = (
            used_scalar
        )

        pool.idx = transcript_indices
        pool.true = true_scalar
        pool.used = used_scalar

        # Intrinsic cargo payload follows authentic
        # transcript-associated material identity.
        authentic_d2 = np.mean(
            authentic_material[
                :,
                :min(3, D)
            ] ** 2,
            axis=1
        )

        payload = np.exp(
            -0.5
            * authentic_d2
            / ell ** 2
        )

        strong = (
            payload
            >= np.quantile(
                payload,
                1.0
                - strong_fraction
            )
        )

        # The model receives model-used material for
        # docking, but authentic payload identity.
        return (
            model_used_material,
            payload.astype(float),
            strong.astype(bool),
        )

    pool.idx = None
    pool.true = None
    pool.used = None

    return pool

def run_one(plan,seed_index):
    score_name=plan['score_definition']; pool=make_pool(COORDINATES[score_name],plan['anchor'],plan['mode'])
    M.make_material_pool=pool
    parameter_seed=PARAMETER_SEED_BASE+seed_index; simulation_seed=SIMULATION_SEED_BASE+seed_index
    try:
        parameters=sample_params(parameter_seed,CFG)
        output=simulate_one(parameters,seed=simulation_seed,variant_name=MODEL_VARIANT)
    finally:M.make_material_pool=ORIGINAL_MATERIAL_POOL
    labels=np.asarray([LABELS[int(i)] for i in pool.idx]); score_map=scores[score_name].to_dict()
    granules=pd.DataFrame({'granule':np.arange(len(labels)),'transcript_label':labels,
                           'analysis_group':[GROUP_MAP[x] for x in labels],
                           'source_score':[score_map[x] for x in labels],
                           'true_coordinate':pool.true,'model_used_coordinate':pool.used})
    supply=granules.groupby(['transcript_label','analysis_group','source_score'],as_index=False).agg(
        supply_count=('granule','size'),true_coordinate=('true_coordinate','mean'),
        model_used_coordinate=('model_used_coordinate','mean'))
    docks=output['dock_df'].copy()
    if len(docks):
        lookup=granules.set_index('granule'); docks['transcript_label']=docks['granule'].map(lookup['transcript_label'])
        docks['total_docks']=1; docks['active_docks']=(docks['active_synapse']==1).astype(int)
        docks['total_productive']=docks['productive'].astype(int)
        docks['active_productive']=((docks['active_synapse']==1)&(docks['productive']==1)).astype(int)
        ec=docks.groupby('transcript_label',as_index=False).agg(
            total_docks=('total_docks','sum'),active_docks=('active_docks','sum'),
            total_productive=('total_productive','sum'),active_productive=('active_productive','sum'))
    else:
        ec=pd.DataFrame(columns=['transcript_label','total_docks','active_docks','total_productive','active_productive'])
    tx=supply.merge(ec,on='transcript_label',how='left')
    count_cols=['total_docks','active_docks','total_productive','active_productive']
    tx[count_cols]=tx[count_cols].fillna(0).astype(int)
    meta={'dataset':DATASET_ID,'panel_hash':PANEL_HASH,'variant':MODEL_VARIANT,'observable':plan['observable'],'analysis_name':plan['analysis_name'],
          'score_definition':score_name,'mode':plan['mode'],'psd_state':plan['psd_state'],'anchor':plan['anchor'],
          'seed_index':seed_index,'parameter_seed':parameter_seed,'simulation_seed':simulation_seed}
    for key,value in meta.items():tx[key]=value
    summary={**meta,'n_granules':int(tx['supply_count'].sum()),'n_unique_transcripts':len(tx),
             'total_docks':int(tx['total_docks'].sum()),'active_docks':int(tx['active_docks'].sum()),
             'total_productive':int(tx['total_productive'].sum()),'active_productive':int(tx['active_productive'].sum())}
    return tx,pd.DataFrame([summary])

smoke=next(row for row in RUN_PLAN if row['observable']=='tau2_ms' and row['analysis_name']=='primary_g_rg4' and row['psd_state']=='WT_like')
smoke_tx,smoke_summary=run_one(smoke,999)
assert smoke_summary['total_docks'].iloc[0]>0 and smoke_tx['analysis_group'].notna().all()
print(smoke_summary.T)

## 13. Resumable 10-seed chunks

Rerun this cell after a disconnect. Valid completed chunks are skipped.

In [ ]:
def safe(text):return re.sub(r'[^A-Za-z0-9_.-]+','_',str(text))
def paths_for(plan,start,stop):
    stem='__'.join([PANEL_HASH[:12],safe(plan['observable']),safe(plan['analysis_name']),safe(plan['psd_state']),f'{start:03d}_{stop-1:03d}'])
    return DIRS['checkpoints']/f'{stem}__summary.csv',DIRS['checkpoints']/f'{stem}__transcripts.csv.gz'
def atomic_csv(df,path,compression=None):
    temporary=path.with_name(path.name+'.tmp'); df.to_csv(temporary,index=False,float_format=FLOAT_FORMAT,compression=compression); os.replace(temporary,path)
def valid_chunk(sp,tp,seeds):
    if not sp.exists() or not tp.exists():return False
    try:
        table=pd.read_csv(sp); return len(table)==len(seeds) and set(table['seed_index'].astype(int))==set(seeds) and set(table['panel_hash'])=={PANEL_HASH}
    except Exception:return False

for number,plan in enumerate(RUN_PLAN,1):
    print(f'Condition {number}/{len(RUN_PLAN)}',plan)
    for start in range(0,int(plan['n_seeds']),CHUNK_SIZE):
        stop=min(start+CHUNK_SIZE,int(plan['n_seeds'])); seeds_chunk=list(range(start,stop)); sp,tp=paths_for(plan,start,stop)
        if valid_chunk(sp,tp,seeds_chunk):
            print(' Reusing',start,'-',stop-1);continue
        tx_frames=[];sum_frames=[]
        print(' Running',start,'-',stop-1)
        try:
            for seed_index in seeds_chunk:
                tx,summary=run_one(plan,seed_index);tx_frames.append(tx);sum_frames.append(summary)
            atomic_csv(pd.concat(sum_frames,ignore_index=True),sp)
            atomic_csv(pd.concat(tx_frames,ignore_index=True),tp,'gzip')
        except Exception as error:
            failure={**plan,'start':start,'stop':stop,'error':repr(error),'time':datetime.datetime.now().isoformat()}
            with open(DIRS['logs']/'failure.json','w') as handle:json.dump(failure,handle,indent=2)
            raise
print('All chunks complete.')

## 14. Aggregate checkpoints and compute group preferences

In [ ]:
summary_files=sorted(DIRS['checkpoints'].glob('*__summary.csv'))
tx_files=sorted(DIRS['checkpoints'].glob('*__transcripts.csv.gz'))
seed_summary=pd.concat([pd.read_csv(path) for path in summary_files],ignore_index=True)
transcript_counts=pd.concat([pd.read_csv(path,compression='gzip') for path in tx_files],ignore_index=True)
key=['observable','analysis_name','psd_state','seed_index']
assert not seed_summary.duplicated(key).any()
expected=int(sum(row['n_seeds'] for row in RUN_PLAN));assert len(seed_summary)==expected
seed_summary.to_csv(DIRS['data']/'perez_seed_level_counts.csv',index=False,float_format=FLOAT_FORMAT)
transcript_counts.to_csv(DIRS['data']/'perez_transcript_seed_counts.csv.gz',index=False,float_format=FLOAT_FORMAT,compression='gzip')

condition=['dataset','observable','analysis_name','score_definition','mode','psd_state','anchor','seed_index','parameter_seed','simulation_seed']
group_counts=transcript_counts.groupby(condition+['analysis_group'],as_index=False).agg(
    supply=('supply_count','sum'),physical_docking=('active_docks','sum'),productive_capture=('active_productive','sum'))
base=seed_summary[condition].drop_duplicates(); preference=[]
for outcome in ['physical_docking','productive_capture']:
    sub=group_counts[condition+['analysis_group','supply',outcome]]
    d=sub[sub['analysis_group']=='dendritic_enriched'].drop(columns='analysis_group').rename(columns={'supply':'d_supply',outcome:'d_events'})
    s=sub[sub['analysis_group']=='soma_enriched_matched'].drop(columns='analysis_group').rename(columns={'supply':'s_supply',outcome:'s_events'})
    table=base.merge(d,on=condition,how='left').merge(s,on=condition,how='left')
    for column in ['d_supply','s_supply','d_events','s_events']:table[column]=table[column].fillna(0)
    table['d_rate']=(table['d_events']+0.5)/(table['d_supply']+1)
    table['s_rate']=(table['s_events']+0.5)/(table['s_supply']+1)
    table['preference_fraction']=table['d_rate']/(table['d_rate']+table['s_rate'])
    table['log_rate_ratio']=np.log(table['d_rate']/table['s_rate']);table['outcome']=outcome;preference.append(table)
group_preferences=pd.concat(preference,ignore_index=True)
assert np.isfinite(group_preferences[['preference_fraction','log_rate_ratio']]).all().all()
group_preferences.to_csv(DIRS['data']/'perez_group_preferences_by_seed.csv',index=False,float_format=FLOAT_FORMAT)
print(seed_summary.groupby(['observable','analysis_name','psd_state'])['seed_index'].nunique())

## 15. WT−ME shifts, statistical summaries, and contrasts

In [ ]:
pair_index=['dataset','observable','analysis_name','score_definition','mode','outcome','seed_index','parameter_seed','simulation_seed']
frac=group_preferences.pivot_table(index=pair_index,columns='psd_state',values='preference_fraction',aggfunc='first').reset_index()
logr=group_preferences.pivot_table(index=pair_index,columns='psd_state',values='log_rate_ratio',aggfunc='first').reset_index()
assert not frac[['WT_like','ME_like']].isna().any().any() and not logr[['WT_like','ME_like']].isna().any().any()
shifts=frac[pair_index].copy();shifts['WT_preference_fraction']=frac['WT_like'];shifts['ME_preference_fraction']=frac['ME_like']
shifts['shift_preference_fraction']=frac['WT_like']-frac['ME_like'];shifts['WT_log_rate_ratio']=logr['WT_like'];shifts['ME_log_rate_ratio']=logr['ME_like'];shifts['shift_log_rate_ratio']=logr['WT_like']-logr['ME_like']
shifts.to_csv(DIRS['data']/'perez_wt_me_seed_shifts.csv',index=False,float_format=FLOAT_FORMAT)

summary_rows=[];groups=['dataset','observable','analysis_name','score_definition','mode','outcome']
for group_key,sub in shifts.groupby(groups):
    for metric in ['shift_preference_fraction','shift_log_rate_ratio']:
        result=inference(sub[metric],seed=20261100+len(summary_rows));result.update(dict(zip(groups,group_key)));result['metric']=metric;summary_rows.append(result)
shift_summary=pd.DataFrame(summary_rows);shift_summary['q_bh_global']=multipletests(shift_summary['empirical_p'],method='fdr_bh')[1]
shift_summary.to_csv(DIRS['statistics']/'perez_wt_me_shift_summary.csv',index=False,float_format=FLOAT_FORMAT)

contrast_rows=[]
for (observable,outcome),sub in shifts.groupby(['observable','outcome']):
    for metric in ['shift_preference_fraction','shift_log_rate_ratio']:
        matrix=sub.pivot_table(index='seed_index',columns='analysis_name',values=metric,aggfunc='first')
        required=['g_only','primary_g_rg4','primary_mapping_shuffled']
        assert not matrix[required].isna().any().any()
        contrasts={'g_only_minus_shuffled':matrix['g_only']-matrix['primary_mapping_shuffled'],
                   'primary_minus_shuffled':matrix['primary_g_rg4']-matrix['primary_mapping_shuffled'],
                   'g_only_minus_primary':matrix['g_only']-matrix['primary_g_rg4']}
        for name,values in contrasts.items():
            result=inference(values,seed=20261200+len(contrast_rows));result.update({'dataset':'Perez_2021_GSE157204','observable':observable,'outcome':outcome,'metric':metric,'contrast':name});contrast_rows.append(result)
contrasts=pd.DataFrame(contrast_rows);contrasts['q_bh_global']=multipletests(contrasts['empirical_p'],method='fdr_bh')[1]
contrasts.to_csv(DIRS['statistics']/'perez_mapping_contrasts.csv',index=False,float_format=FLOAT_FORMAT)
display(shift_summary.sort_values(['observable','outcome','metric','estimate']))
display(contrasts.sort_values(['observable','outcome','metric','estimate']))

## 16. Export plotting-ready long tables for R

In [ ]:
sequence_effects.to_csv(DIRS['plotting']/'panel_B_perez_sequence_effects.csv',index=False,float_format=FLOAT_FORMAT)

state_rows=[];state_groups=['dataset','observable','analysis_name','score_definition','mode','outcome','psd_state']
for group_key,sub in group_preferences.groupby(state_groups):
    for metric in ['preference_fraction','log_rate_ratio']:
        values=sub[metric].to_numpy(float);rng=np.random.default_rng(20261300+len(state_rows));idx=rng.integers(0,len(values),size=(N_BOOT,len(values)));boot=values[idx].mean(1)
        row=dict(zip(state_groups,group_key));row.update({'metric':metric,'n_seeds':len(values),'estimate':float(values.mean()),'ci_low':float(np.percentile(boot,2.5)),'ci_high':float(np.percentile(boot,97.5))});state_rows.append(row)
state_preferences=pd.DataFrame(state_rows)
state_preferences.to_csv(DIRS['plotting']/'panel_C_perez_state_preferences.csv',index=False,float_format=FLOAT_FORMAT)
shift_summary.to_csv(DIRS['plotting']/'panel_D_perez_wt_me_shifts.csv',index=False,float_format=FLOAT_FORMAT)
contrasts.to_csv(DIRS['plotting']/'panel_D_perez_mapping_contrasts.csv',index=False,float_format=FLOAT_FORMAT)
shift_summary.to_csv(DIRS['plotting']/'perez_psd_observable_sensitivity.csv',index=False,float_format=FLOAT_FORMAT)

print('Plotting tables:')
for path in sorted(DIRS['plotting'].glob('*.csv')):print(path.name,round(path.stat().st_size/1e3,1),'KB')

## 17. Final audit and bundle

In [ ]:
# 17. Final audit, R inputs, manifest and single reproducibility ZIP
coverage = seed_summary.groupby(['observable','analysis_name','psd_state'])['seed_index'].nunique().to_dict()
expected_coverage = {(obs, analysis, state): n
                     for obs, n in OBSERVABLES.items()
                     for analysis in ANALYSES
                     for state in ['WT_like', 'ME_like']}
coverage_ok = all(coverage.get(key) == value for key, value in expected_coverage.items())

finite_ok = (
    np.isfinite(group_preferences[['preference_fraction','log_rate_ratio']]).all().all()
    and np.isfinite(shifts[['shift_preference_fraction','shift_log_rate_ratio']]).all().all()
    and np.isfinite(shift_summary[['estimate','ci_low','ci_high','empirical_p']]).all().all()
)

canonical_names = [
    'panel_B_perez_sequence_effects.csv',
    'panel_C_perez_state_preferences.csv',
    'panel_D_perez_wt_me_shifts.csv',
    'panel_D_perez_mapping_contrasts.csv',
]
found = {}
for name in canonical_names:
    path = DIRS['plotting'] / name
    assert path.exists() and path.stat().st_size > 0, f'Canonical table missing or empty: {path}'
    found[name] = path

sequence_effects_final = pd.read_csv(found['panel_B_perez_sequence_effects.csv'])
state_preferences_final = pd.read_csv(found['panel_C_perez_state_preferences.csv'])
wt_me_shifts_final = pd.read_csv(found['panel_D_perez_wt_me_shifts.csv'])
mapping_contrasts_final = pd.read_csv(found['panel_D_perez_mapping_contrasts.csv'])

assert set(sequence_effects_final['dataset']) == {DATASET_ID}
assert len(sequence_effects_final) == len(score_names)
assert set(sequence_effects_final['n']) == {N_MATCHED_PER_GROUP}
assert set(state_preferences_final['n_seeds']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert set(wt_me_shifts_final['n']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert set(mapping_contrasts_final['n']) == {SENSITIVITY_N_SEEDS, PRIMARY_N_SEEDS}
assert {'primary_g_rg4','g_only','primary_mapping_shuffled'}.issubset(
    set(wt_me_shifts_final['analysis_name']))

for name, path in found.items():
    shutil.copy2(path, R_INPUT / name)

supplementary_inputs = [
    (DIRS['panels'], 'perez_frozen_matched_panel.csv'),
    (DIRS['panels'], 'perez_matching_balance.csv'),
    (DIRS['panels'], 'perez_pair_diagnostics.csv'),
    (DIRS['panels'], 'perez_support_diagnostics.csv'),
    (DIRS['panels'], 'perez_retained_pairs_trim95.csv'),
    (DIRS['panels'], 'perez_frozen_panel_trim95.csv'),
    (DIRS['statistics'], 'perez_sequence_sparsity.csv'),
    (DIRS['statistics'], 'perez_sequence_effects_trim95.csv'),
    (DIRS['statistics'], 'perez_trim95_concordance.csv'),
]
for directory, name in supplementary_inputs:
    path = directory / name
    if path.exists() and path.stat().st_size > 0:
        shutil.copy2(path, R_INPUT / name)
        print('Copied:', name)
    else:
        print('Missing supplementary table:', name)

outputs_ok = all(path.exists() and path.stat().st_size > 0 for path in found.values())
if not all([coverage_ok, finite_ok, outputs_ok]):
    raise RuntimeError(
        f'Final audit failed: coverage={coverage_ok} finite={finite_ok} outputs={outputs_ok}')

manifest = {
    'dataset': DATASET_ID,
    'species': SPECIES,
    'assembly': RAT_ASSEMBLY,
    'ensembl_release': ENSEMBL_RELEASE,
    'analysis_unit': ANALYSIS_UNIT,
    'group_assignment': SELECTION_SOURCE,
    'panel_hash': PANEL_HASH,
    'canonical_source_directory': str(DIRS['plotting']),
    'model_sha256': sha256_file(MODEL_PATH),
    'config_sha256': sha256_file(CFG_PATH),
    'model_variant': MODEL_VARIANT,
    'psd_gain': PSD_GAIN,
    'psd_separation': PSD_SEPARATION,
    'psd_reference_values': JIA,
    'invariant_layer': {
        'n_matched_per_group': N_MATCHED_PER_GROUP,
        'tolerance_grid': list(TOLERANCE_GRID),
        'smallest_feasible_tolerance': tolerance_used,
        'score_names': score_names,
        'n_boot': N_BOOT,
        'n_permutations': N_PERMUTATIONS,
        'primary_n_seeds': PRIMARY_N_SEEDS,
        'sensitivity_n_seeds': SENSITIVITY_N_SEEDS,
        'parameter_seed_base': PARAMETER_SEED_BASE,
        'simulation_seed_base': SIMULATION_SEED_BASE,
        'trim_quantile': TRIM_QUANTILE,
    },
    'validation': {
        'coverage_ok': bool(coverage_ok),
        'finite_ok': bool(finite_ok),
        'expected_simulations': int(expected),
        'observed_simulations': int(len(seed_summary)),
        'panel_size': int(len(final_panel)),
        'trim95_pairs': int(len(retained_pairs)),
        'sequence_effect_rows': int(len(sequence_effects_final)),
        'state_seed_counts': sorted(state_preferences_final['n_seeds'].unique().tolist()),
        'shift_seed_counts': sorted(wt_me_shifts_final['n'].unique().tolist()),
        'canonical_tables': canonical_names,
    },
    'completed_at': datetime.datetime.now().isoformat(),
    'outputs': {str(path.relative_to(ROOT)): sha256_file(path)
                for path in sorted(ROOT.rglob('*')) if path.is_file()},
}
(ROOT / 'MANIFEST_perez.json').write_text(json.dumps(manifest, indent=2))

anchor_text = {key: round(float(value), 6) for key, value in PSD_SEPARATION.items()}

readme = f"""# MSC Figure 6 Perez reproducibility bundle

Dataset: Perez et al. 2021, GEO GSE157204, published soma/dendrite labels from
Supplementary File 5. Annotation: Ensembl release {ENSEMBL_RELEASE}, {SPECIES},
{RAT_ASSEMBLY}. Analysis unit: gene-level canonical 3'UTR.

Group assignment uses the published discrete Dendrites/Somata classification,
filtered on adjusted p below 0.05 and the sign of the average log fold change.
Group assignment and matching are sequence-blind.

Invariant analysis layer, shared with the Middleton and Tushev workflows:
{N_MATCHED_PER_GROUP} genes per group, nine score definitions, {N_BOOT} bootstrap
resamples, {N_PERMUTATIONS} sign-flip permutations with Benjamini-Hochberg
correction, and an 18-condition simulation grid using {PRIMARY_N_SEEDS} seeds for
the primary tau2 PSD mapping and {SENSITIVITY_N_SEEDS} seeds each for elastic
modulus and mobile fraction.

PSD anchors are calibrated from published wild-type versus oligomerisation-
deficient PSD contrasts with gain {PSD_GAIN}, giving {anchor_text}. WT-like uses
the positive anchor and ME-like the negative anchor.

Simulations call msc_model_feedback.simulate_one with variant {MODEL_VARIANT}.
The transcript-informed material pool temporarily replaces make_material_pool.
Docking receives the model-used material coordinate while the productive payload
follows the authentic transcript coordinate, so true and used coordinates stay
separable and the shuffled-mapping control is not tautological.

Checkpoints are keyed by the panel hash {PANEL_HASH[:12]}.

A Trim-95 sensitivity analysis repeats the paired sequence-effect inference on the
95 percent best-matched pairs. Sequence-feature sparsity is reported separately
because strict rG4 sites are largely absent from this dataset.

Canonical tables consumed by the three-dataset plotting workflow are copied from
plotting/ into 10_r_input/.

All docking and productive-capture quantities are stochastic in-silico predictions
of the MSC model, not wet-lab measurements.
"""
(ROOT / 'README.md').write_text(readme)

shutil.make_archive(str(ROOT / 'perez_R_plotting_tables'), 'zip', root_dir=DIRS['plotting'])

zip_path = Path('/content/MSC_Figure6_Perez_reproducible_bundle.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT.parent))

assert zip_path.exists() and zip_path.stat().st_size > 0
if USE_DRIVE_MIRROR:
    shutil.copy2(zip_path, DRIVE_MIRROR / zip_path.name)

print('\nFINAL AUDIT PASSED')
print('Panel hash:', PANEL_HASH)
print('ZIP:', zip_path.name, f'{zip_path.stat().st_size / 1e6:.1f} MB')
print('\nCanonical R inputs:')
for name in canonical_names:
    print('-', R_INPUT / name)
display(sequence_effects_final)
files.download(str(zip_path))
